# Prosody Extraction for 555 Videos

**Purpose:** Extract 21-dim prosody for the SAME 555 videos that have WavLM embeddings.

**Data alignment confirmed:**
- WavLM: 555 videos (wavlm_utterance_safe) ✅
- Labels: 555 videos (utterances_clean.jsonl) ✅
- Audio: 555 videos (chuckle_audio folders) ✅

**Pipeline:**
1. Mount Google Drive
2. Download utterances_clean.jsonl (if not present)
3. For each of 555 WavLM videos:
   - Get audio file path
   - For each utterance (start, end):
     - Extract prosody: F0(5) + Energy(5) + Duration(2) + Spectral(5) + Voice(4) = 21-dim
4. Save prosody_555.json

In [ ]:
# @title Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
# @title Step 2: Imports & Setup
import os
import json
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
import time
import subprocess

BASE_DIR = Path('/content/gdrive/MyDrive')
WAVLM_DIR = BASE_DIR / 'wavlm_utterance_safe'
OUTPUT_FILE = BASE_DIR / 'prosody_555.json'

SR = 16000  # Sample rate

# Audio folder search order
AUDIO_FOLDERS = [
    'chuckle_audio',
    'chuckle_audio_all/audio',
    'chuckle_audio_all/audio_final',
    'chuckle_audio_all/audio_new',
    'chuckle_audio_all/audio_all'
]

In [ ]:
# @title Step 3: Download utterances_clean.jsonl
# Check if file exists in various locations
possible_utt_paths = [
    BASE_DIR / 'utterances_clean.jsonl',
    Path('/content/utterances_clean.jsonl'),
    Path('/content/gdrive/utterances_clean.jsonl'),
    Path('/content/gdrive/MyDrive/utterances_clean.jsonl')
]

utt_path = None
for p in possible_utt_paths:
    if p.exists():
        utt_path = p
        print(f'Found utterances at: {p}')
        break

if utt_path is None:
    print('Downloading utterances_clean.jsonl from Google Drive root...')
    # File is at Google Drive root, download it
    subprocess.run([
        'rclone', 'copy',
        'gdrive:utterances_clean.jsonl',
        '/content/'
    ], check=True)
    utt_path = Path('/content/utterances_clean.jsonl')
    print(f'Downloaded to: {utt_path}')

print(f'Using: {utt_path}')

In [ ]:
# @title Step 4: Load WavLM video IDs
# Get the 555 video IDs that have WavLM
wavlm_files = list(WAVLM_DIR.glob('*.json'))
wavlm_files = [f for f in wavlm_files if f.name != 'checkpoint.json']

wavlm_video_ids = set()
for f in wavlm_files:
    with open(f) as fh:
        data = json.load(fh)
        wavlm_video_ids.add(data['video_id'])

print(f'WavLM videos: {len(wavlm_video_ids)}')
print(f'Sample: {list(wavlm_video_ids)[:3]}')

In [ ]:
# @title Step 5: Build audio path lookup
# For each audio folder, build a map of video_id -> full_path
audio_paths = {}  # video_id -> path

for folder in AUDIO_FOLDERS:
    folder_path = BASE_DIR / folder
    if not folder_path.exists():
        print(f'  {folder}: NOT FOUND')
        continue
    
    files = list(folder_path.iterdir())
    count = 0
    for f in files:
        if f.suffix in ['.wav', '.mp3', '.m4a'] and not f.name.endswith('.part'):
            vid = f.stem  # filename without extension
            if vid not in audio_paths:  # Prefer earlier folders
                audio_paths[vid] = str(f)
                count += 1
    print(f'  {folder}: {count} audio files')

print(f'\nTotal unique audio: {len(audio_paths)}')
print(f'WavLM videos with audio: {len(wavlm_video_ids & set(audio_paths.keys()))}')

In [ ]:
# @title Step 6: Load utterances for WavLM videos
from collections import defaultdict

print(f'Loading utterances from: {utt_path}')

# Load utterances for WavLM videos only
utterances_by_video = defaultdict(list)
total_utts = 0

with open(utt_path) as f:
    for line in f:
        d = json.loads(line)
        vid = d['video_id']
        if vid in wavlm_video_ids:
            utterances_by_video[vid].append({
                'start': d['start'],
                'end': d['end'],
                'label': d.get('label', 0),
                'text': d.get('text', '')
            })
            total_utts += 1

print(f'Videos with utterances: {len(utterances_by_video)}')
print(f'Total utterances: {total_utts}')

In [ ]:
# @title Step 7: Prosody extraction function
def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features."""
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    try:
        rms = librosa.feature.rms(y=y)[0]
        features.extend([
            np.mean(rms),
            np.std(rms),
            np.max(rms),
            np.min(rms),
            np.max(rms) - np.min(rms)
        ])
    except:
        features.extend([0]*5)
    
    # Duration - 2 dims
    try:
        duration = len(y) / sr
        speech_rate = np.sum(rms > np.mean(rms)) / duration if duration > 0 else 0
        features.extend([duration, speech_rate])
    except:
        features.extend([0]*2)
    
    # Spectral - 5 dims
    try:
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        spec_flat = librosa.feature.spectral_flatness(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features.extend([
            np.mean(spec_cent),
            np.mean(spec_bw),
            np.mean(spec_flat),
            np.mean(zcr),
            np.std(zcr)
        ])
    except:
        features.extend([0]*5)
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
        features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    except:
        features.extend([0]*4)
    
    return np.array(features, dtype=np.float32)

In [ ]:
# @title Step 8: Extract prosody for all 555 videos
print('Starting prosody extraction for 555 videos...')
t0 = time.time()

prosody_data = {}  # video_id -> list of {start, end, prosody, label}
failed_videos = []
total_utts = 0

for vid in tqdm(wavlm_video_ids, desc='Videos'):
    # Get audio path
    audio_path = audio_paths.get(vid)
    if not audio_path:
        failed_videos.append(vid)
        continue
    
    # Load audio
    try:
        if audio_path.endswith('.wav'):
            y, sr = sf.read(audio_path, dtype='float32')
        else:
            y, sr = librosa.load(audio_path, sr=SR, mono=True)
        
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        if sr != SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    except Exception as e:
        print(f'\nError loading {vid}: {e}')
        failed_videos.append(vid)
        continue
    
    # Get utterances for this video
    utts = utterances_by_video.get(vid, [])
    if not utts:
        failed_videos.append(vid)
        continue
    
    video_prosody = []
    for utt in utts:
        start_s = utt['start']
        end_s = utt['end']
        
        start_sample = int(start_s * SR)
        end_sample = int(end_s * SR)
        
        if end_sample > len(y):
            end_sample = len(y)
        if start_sample >= len(y):
            start_sample = 0
        
        y_slice = y[start_sample:end_sample]
        
        if len(y_slice) < SR * 0.1:  # Less than 100ms
            video_prosody.append({
                'start': start_s,
                'end': end_s,
                'prosody': np.zeros(21, dtype=np.float32).tolist(),
                'label': utt['label']
            })
        else:
            prosody = extract_prosody_21dim(y_slice, SR)
            video_prosody.append({
                'start': start_s,
                'end': end_s,
                'prosody': prosody.tolist(),
                'label': utt['label']
            })
        
        total_utts += 1
    
    prosody_data[vid] = video_prosody

elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')
print(f'Videos processed: {len(prosody_data)}/{len(wavlm_video_ids)}')
print(f'Failed videos: {len(failed_videos)}')
print(f'Total utterances: {total_utts}')

In [ ]:
# @title Step 9: Save prosody data
print(f'Saving to {OUTPUT_FILE}...')

# Convert to JSON-serializable format
output = {
    'video_count': len(prosody_data),
    'total_utterances': total_utts,
    'prosody': prosody_data
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f)

print(f'Saved!')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1e6:.1f} MB')

In [ ]:
# @title Step 10: Verify
print('=== VERIFICATION ===')

# Check output file exists
if OUTPUT_FILE.exists():
    size_mb = OUTPUT_FILE.stat().st_size / 1e6
    print(f'✅ Output file exists: {size_mb:.1f} MB')
else:
    print('❌ Output file NOT found!')

# Check a sample
with open(OUTPUT_FILE) as f:
    data = json.load(f)

print(f"\nVideos: {data['video_count']}")
print(f"Total utterances: {data['total_utterances']}")

# Sample
sample_vid = list(data['prosody'].keys())[0]
sample_utts = data['prosody'][sample_vid][:2]
print(f"\nSample video: {sample_vid}")
print(f"Sample utterances: {len(sample_utts)}")
if sample_utts:
    print(f"First utt prosody dims: {len(sample_utts[0]['prosody'])}")
    print(f"First utt label: {sample_utts[0]['label']}")

# Count positive
pos_count = sum(
    utt['label'] 
    for utts in data['prosody'].values() 
    for utt in utts
)
total = data['total_utterances']
print(f"\nPositive rate: {pos_count}/{total} ({pos_count/total*100:.1f}%)")

print('\n✅ Extraction complete!')